# 01 — The Five-Trace Phone Test
## Why Local Pass $\neq$ Global Pass: A Hand-Calculated Introduction

**Series:** `001_finite_obstruction_calculus` / `012_lggr_i_evaluated_path_kernels`  
**Author:** Jeremy H. Carroll  
**Status:** Living Educational Supplement  
**Anchor Tag:** `sealed-derivation-v1` (`3100c4e`, cut 2026-08-17)

---

> ### **Claim-Typing Legend**
> - **[Pinned]**: Verified directly by executable Python functions in `code/epk_core.py` and passing test suites.
> - **[Proved]**: Mechanized in Lean 4 under `PaperAE004` (reported kernel axioms: `propext`, `Classical.choice`, `Quot.sound` or axiom-free).
> - **[Declared]**: An operational definition, architectural parameter, or illustrative teaching model chosen by the author.
> - **[Open]**: A mathematical or empirical question not discharged in this package.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. The Real-World Intuition

Imagine you have a software pipeline powered by two cooperating LLM agents:

- **Stage 1 (Agent 1):** Generates a SQL migration script.  
  - *Local Validator $E_1$:* Checks if the SQL has valid syntax and compiles against the database schema.  
- **Stage 2 (Agent 2):** Generates a backend Python function calling the database.  
  - *Local Validator $E_2$:* Checks if the Python code has valid syntax and passes type checking.  
- **Integration Test $E_g$:** Deploys both changes to a staging environment and executes an end-to-end API query.

### The Stealth Failure
Suppose Agent 1 renames a database column from `user_id` to `account_id` (valid SQL, $E_1 = 1$).  
Agent 2 writes code that queries `user_id` (valid Python syntax, $E_2 = 1$).  
Both local checkers report **`GREEN`** ($E_1 = 1$ and $E_2 = 1$).  
When the integration test runs, the query fails with `ColumnNotFound`! **`RED`** ($E_g = 0$).

This is the canonical failure mode:
$$E_1 = 1, \quad E_2 = 1, \quad E_g = 0$$

### The Evaluator Gluing Defect ($\Delta_E$)
To measure this failure mathematically, we define the **Evaluator Gluing Defect**:
$$\Delta_E = E_g - (E_1 \cdot E_2)$$

In our stealth failure:
$$\Delta_E = 0 - (1 \cdot 1) = -1$$

A defect of **$-1$** is called a **Negative Gluing Defect (NGD)**. It means the local validators overpromised success: they claimed green, but the real task was red!


## 2. The Canonical Five-Trace Phone Test (§2.2)

In Section 2.2 of the paper, we present the **Five-Trace Phone Test**: the simplest possible sample that exhibits evaluator non-compositionality. 

Let's calculate $\\Delta_E$ by hand for each of the 5 traces:

| Trace ID | $E_1$ (Stage 1) | $E_2$ (Stage 2) | Local Product ($E_1 E_2$) | $E_g$ (Integration) | $\\Delta_E = E_g - E_1 E_2$ | Defect Category |
| :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| **1** | $1$ (Pass) | $1$ (Pass) | $1 \\times 1 = 1$ | $1$ (Pass) | $1 - 1 = 0$ | **ZDR** (Aligned Pass) |
| **2** | $1$ (Pass) | $1$ (Pass) | $1 \\times 1 = 1$ | $0$ (Fail) | $0 - 1 = -1$ | **NGD** (Local Green, Global Red) |
| **3** | $1$ (Pass) | $1$ (Pass) | $1 \\times 1 = 1$ | $0$ (Fail) | $0 - 1 = -1$ | **NGD** (Local Green, Global Red) |
| **4** | $1$ (Pass) | $0$ (Fail) | $1 \\times 0 = 0$ | $0$ (Fail) | $0 - 0 = 0$ | **ZDR** (Aligned Failure) |
| **5** | $0$ (Fail) | $1$ (Pass) | $0 \\times 1 = 0$ | $0$ (Fail) | $0 - 0 = 0$ | **ZDR** (Aligned Failure) |

Notice:
- In Traces 4 and 5, at least one local step failed, so the local product $E_1 E_2 = 0$. The global test also failed ($E_g = 0$). So local and global agree ($\Delta_E = 0$).
- Traces 2 and 3 are the culprits: both local steps passed, but global failed ($\Delta_E = -1$).


## 3. Summary Statistics: Hand Calculation

Now let's compute the sample-level rates across the $N = 5$ traces:

1. **Local Pass Rate (LPR):** How often do both local validators pass?
   $$\mathrm{LPR} = \frac{1 + 1 + 1 + 0 + 0}{5} = \frac{3}{5} = 0.60 \quad (60\%)$$

2. **Global Pass Rate (GPR):** How often does the actual task succeed?
   $$\mathrm{GPR} = \frac{1 + 0 + 0 + 0 + 0}{5} = \frac{1}{5} = 0.20 \quad (20\%)$$

3. **Mean Defect ($\bar{\Delta}_E$):** What is the average value of $\Delta_E$?
   $$\bar{\Delta}_E = \frac{0 + (-1) + (-1) + 0 + 0}{5} = \frac{-2}{5} = -0.40$$

4. **Defect Distribution Rates:**
   - **Negative Gluing Defect (NGD) Rate:** Fraction with $\Delta_E = -1$:  
     $$\mathrm{NGD} = \frac{2}{5} = 0.40 \quad (40\%)$$
   - **Positive Gluing Defect (PGD) Rate:** Fraction with $\Delta_E = +1$:  
     $$\mathrm{PGD} = \frac{0}{5} = 0.00 \quad (0\%)$$
   - **Zero Defect Residual (ZDR) Rate:** Fraction with $\Delta_E = 0$:  
     $$\mathrm{ZDR} = \frac{3}{5} = 0.60 \quad (60\%)$$


In [ ]:
# Load the canonical five-trace fixture from epk_core
phone_traces = make_five_trace_phone_test()

# Compute metrics using the production engine
metrics = EPKMetrics.compute(phone_traces)

print("=== FIVE-TRACE PHONE TEST METRICS ===")
print(f"Sample Size (N)             : {metrics.sample_size}")
print(f"Local Pass Rate (LPR)       : {metrics.lpr:.2f} (Hand calculated: 0.60)")
print(f"Global Pass Rate (GPR)      : {metrics.gpr:.2f} (Hand calculated: 0.20)")
print(f"Mean Defect (bar(Delta)_E)   : {metrics.mean_delta_e:.2f} (Hand calculated: -0.40)")
print(f"Negative Defect Rate (NGD)  : {metrics.ngd_all:.2f} (Hand calculated: 0.40)")
print(f"Positive Defect Rate (PGD)  : {metrics.pgd:.2f} (Hand calculated: 0.00)")
print(f"Zero Defect Rate (ZDR)      : {metrics.zdr:.2f} (Hand calculated: 0.60)")

# Verify exact machine equality
assert metrics.lpr == 0.60
assert metrics.gpr == 0.20
assert metrics.mean_delta_e == -0.40
assert metrics.ngd_all == 0.40
assert metrics.pgd == 0.00
assert metrics.zdr == 0.60
print("\n[SUCCESS] Production engine matches hand calculations to machine precision!")


## 4. The Five Sample Balance Identities

Because $\Delta_E = E_g - E_1 E_2$ is an exact algebraic definition on finite counts, five exact identities hold on **any** sample of traces, regardless of sample size or distribution:

1. **Pass-Rate Gap Identity:**
   $$\mathrm{GPR} - \mathrm{LPR} = \bar{\Delta}_E$$
   *Check:* $0.20 - 0.60 = -0.40$. The local dashboard overestimates true integration success by exactly the mean defect!

2. **Trichotomy Normal Form:**
   $$\mathrm{NGD} + \mathrm{PGD} + \mathrm{ZDR} = 1$$
   *Check:* $0.40 + 0.00 + 0.60 = 1.00$. Every trace falls into exactly one of the three defect categories.

3. **Signed Defect Balance:**
   $$\bar{\Delta}_E = \mathrm{PGD} - \mathrm{NGD}$$
   *Check:* $0.00 - 0.40 = -0.40$. The mean defect is simply the difference between positive and negative defect frequencies.

4. **Count Difference:**
   $$\mathrm{Count}(\Delta_E = +1) - \mathrm{Count}(\Delta_E = -1) = N \cdot \bar{\Delta}_E$$
   *Check:* $0 - 2 = 5 \times (-0.40) = -2$.

5. **Count-Gap Identity:**
   $$N_{\mathrm{global}} - N_{\mathrm{local}} = N_\Delta \quad \text{where} \quad N_\Delta = \sum_{i=1}^N \Delta_{E,i}$$
   *Check:* $1 - 3 = -2$.


In [ ]:
# Verification of the 5 Exact Identities in code
id1_diff = (metrics.gpr - metrics.lpr) - metrics.mean_delta_e
id2_sum = (metrics.ngd_all + metrics.pgd + metrics.zdr) - 1.0
id3_diff = (metrics.pgd - metrics.ngd_all) - metrics.mean_delta_e
id4_diff = (metrics.pgd_count - metrics.ngd_count) - (metrics.sample_size * metrics.mean_delta_e)
id5_diff = (metrics.global_pass_count - metrics.local_pass_count) - metrics.delta_sum

print("=== IDENTITY VERIFICATION RECEIPTS ===")
print(f"Identity 1 (GPR - LPR == bar(Delta)): error = {abs(id1_diff):.1e}")
print(f"Identity 2 (NGD + PGD + ZDR == 1.0) : error = {abs(id2_sum):.1e}")
print(f"Identity 3 (bar(Delta) == PGD - NGD): error = {abs(id3_diff):.1e}")
print(f"Identity 4 (Count+ - Count- == N*bar): error = {abs(id4_diff):.1e}")
print(f"Identity 5 (N_glob - N_loc == N_del): error = {abs(id5_diff):.1e}")

assert abs(id1_diff) < 1e-15
assert abs(id2_sum) < 1e-15
assert abs(id3_diff) < 1e-15
assert abs(id4_diff) < 1e-15
assert abs(id5_diff) == 0
print("\n[FIXTURE-VERIFIED] All five sample balance identities hold identically on this sample!")


In [ ]:
# Visualizing the 5-Trace Breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

# Panel A: Pass Rates
categories = ['Local Pass Rate (LPR)', 'Global Pass Rate (GPR)']
rates = [metrics.lpr, metrics.gpr]
colors = ['#2b5c8f', '#d95f02']

bars = ax1.bar(categories, rates, color=colors, width=0.5, edgecolor='black', alpha=0.85)
ax1.set_ylim(0, 1.0)
ax1.set_ylabel("Rate", fontsize=11, fontweight='bold')
ax1.set_title("(a) The Overestimation Gap (GPR - LPR = -0.40)", fontsize=11, fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Add value labels
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.03, f"{yval:.1%}", ha='center', va='bottom', fontweight='bold')

# Panel B: Defect Trichotomy Distribution
defect_labels = ['ZDR (Delta = 0)', 'NGD (Delta = -1)', 'PGD (Delta = +1)']
defect_counts = [metrics.zdr_count, metrics.ngd_count, metrics.pgd_count]
pie_colors = ['#4daf4a', '#e41a1c', '#377eb8']

ax2.pie(
    defect_counts,
    labels=defect_labels,
    autopct='%1.0f%%',
    colors=pie_colors,
    startangle=140,
    wedgeprops={'edgecolor': 'black', 'linewidth': 1.2}
)
ax2.set_title("(b) Defect Partition (NGD=40%, ZDR=60%, PGD=0%)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Mandatory Claim Hygiene: CAN-Show vs CANNOT-Show

To maintain strict epistemic integrity:

### What this model CAN show:
- Shows that localized validator pass ($E_1 = 1, E_2 = 1$) does **not** logically or arithmetically imply global task success ($E_g = 1$).
- Shows that the Evaluator Gluing Defect $\Delta_E = E_g - E_1 E_2$ exactly measures the difference between local and global evaluations.
- Shows that the five sample balance identities hold unconditionally for any discrete evaluation traces.

### What this model CANNOT show:
- Does **not** show that all AI failures are evaluator defects (agents also fail from bad prompts, context truncation, or execution timeouts).
- Does **not** claim to be an empirical benchmark on real LLMs (this is a minimal illustrative mathematical fixture).
- Does **not** provide a guarantee that a system with $\Delta_E = 0$ is bug-free (a constant evaluator that always outputs $0$ or always outputs $1$ also produces $\Delta_E = 0$, but is completely useless!).


## 6. Student Exercises

Test your understanding with these two micro-exercises.

### Exercise 1: Dynamic Recovery
Suppose an automated retry mechanism is added to our pipeline so that **Trace 2** succeeds at the integration level ($E_g = 1$).
1. What are the new values of $E_1, E_2, E_g$ for Trace 2?
2. What is the new $\Delta_E$ for Trace 2?
3. What are the new sample $\mathrm{LPR}, \mathrm{GPR},$ and $\bar{\Delta}_E$?

<details>
<summary><b>Click to see the solution for Exercise 1</b></summary>

1. $E_1 = 1, E_2 = 1, E_g = 1$.
2. $\Delta_E = E_g - E_1 E_2 = 1 - (1 \times 1) = 0$ (The defect disappears and becomes ZDR!).
3. New values:
   - $\mathrm{LPR} = 3/5 = 0.60$ (unchanged).
   - $\mathrm{GPR} = 2/5 = 0.40$ (increased from 0.20 to 0.40).
   - $\bar{\Delta}_E = \mathrm{GPR} - \mathrm{LPR} = 0.40 - 0.60 = -0.20$.
   - $\mathrm{NGD} = 1/5 = 0.20$ (reduced from 0.40 to 0.20).
</details>

### Exercise 2: Positive Gluing Defect (PGD)
Can you create a trace where a local stage fails ($E_1 = 0$), but the overall task succeeds ($E_g = 1$)?
What is $\Delta_E$, and what real-world engineering pattern does this correspond to?

<details>
<summary><b>Click to see the solution for Exercise 2</b></summary>

- Trace: $E_1 = 0, E_2 = 1, E_g = 1$.
- Local product: $E_1 E_2 = 0 \times 1 = 0$.
- Defect: $\Delta_E = E_g - E_1 E_2 = 1 - 0 = +1$ (**Positive Gluing Defect**).
- Real-world meaning: **Dynamic Fallback or Error Recovery**. For example, Agent 1 failed to query a primary database ($E_1 = 0$), but the system caught the exception, routed the query to a cache or fallback service, and the user's request completed successfully ($E_g = 1$).
</details>


---
## Anchors & Provenance

- **Paper Sections:** §2.1 (Definition and normal forms), §2.2 (Five-trace example), §2.3 (Exact sample identities), §5 (Finite sample identities).
- **Claim Ledger IDs:** B1 (Counterexample), B2 (Five-Trace Phone Test), E1 (NGD/PGD/ZDR partition).
- **Code Functions:** `epk_core.make_five_trace_phone_test()`, `epk_core.calculate_evaluator_defect()`, `epk_core.EPKMetrics.compute()`.
- **Lean 4 Mechanization:**
  - `PaperAE004.local_green_global_red_counterexample` in `lean4/01_local_green_counterexample.lean`
  - `PaperAE004.phone_identity_global_minus_local` in `lean4/01_local_green_counterexample.lean`
  - `PaperAE004.phone_negative_defect_count` in `lean4/01_local_green_counterexample.lean`
- **Next Step:** Open [`02_the_evaluator_defect_trichotomy.ipynb`](02_the_evaluator_defect_trichotomy.ipynb) to explore why tracking only local and global pass rates hides critical failure modes.
